# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 34.0 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 2.8 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "QCRI/Fanar-1-9B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/18.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

# Load Data

In [ ]:
import pandas as pd
data = pd.read_excel('sampled_sentiment_data.xlsx')

In [ ]:
data.head()

,Tweet_id,sentiment,Text
0,1084541247971384960,Negative,لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتل...
1,1080559232817204992,Negative,وش ذنبه يعيش هالحياة ؟ 💔 ذي نهاية الحب اللي اش...
2,1223397441464040960,Negative,@mhm8889 @turkialhussini1 في ناس جاتهم امراض م...
3,1153415090298905088,Negative,أنا بخلص رياضة من هون وبلاقي جارتنا جايبه صحن ...
4,1080145601336173056,Negative,بقيت احس ان المرض في مصر آخرته الموت مش العلاج...


# Zero Shot

In [ ]:
content = '''أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر. صنّف مشاعر الجملة التالية بناءً على نبرتها العاطفية. اختر شعورًا واحدًا فقط من بين: إيجابي، سلبي، أو محايد.
'''

zero_pred = []
for i, text in enumerate(data['Text']):
    prompt = f'''الجملة:
    {text}

    المشاعر المتوقعة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
سلبي,114
إيجابي,61
سلبية,12
إيجابي\n\nالثقة: 95%,7
محايد\n\nالثقة: 95%,4
...,...
"إيجابي/شاكر/ودي\n\n(ملاحظة: رغم أن الجملة تتضمن ذكر ""كورونا""، إلا أن النبرة العامة هي طلب حماية ودعاء، مما يجعلها إيجابية أو وديّة) ولكن بناءً على السياق الأكثر شيوعاً، يمكن تصنيفها كإيجابية لأنها تعبر عن الثقة في الله والطلب للحماية.",1
إيجابي.\n\n(النغمة في هذه الجملة تعبر عن الثقة والراحة، مما يجعلها إيجابية),1
سلبي\n\n(ملاحظة: النص يحتوي على سخرية وتهكم، مما يجعل النبرة سلبية),1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "سلبي" in pr:
    nor_pre.append("Negative")
  elif "سلبية" in pr:
    nor_pre.append("Negative")
  elif "Negative" in pr:
    nor_pre.append("Negative")
  elif "إيجابي" in pr:
    nor_pre.append("Positive")
  elif "إيجابية" in pr:
    nor_pre.append("Positive")
  elif "ايجابي" in pr:
    nor_pre.append("Positive")
  elif "ايجابية" in pr:
    nor_pre.append("Positive")
  elif "محايد" in pr:
    nor_pre.append("Neutral")
  elif "محايدة" in pr:
    nor_pre.append("Neutral")
  elif "محيادي" in pr:
    nor_pre.append("Neutral")
  elif "محاييد" in pr:
    nor_pre.append("Neutral")
  elif "محيود" in pr:
    nor_pre.append("Neutral")
  elif "المحيد" in pr:
    nor_pre.append("Neutral")
  else:
    nor_pre.append("Unclassified")

pred_zero['Normalized Sentiment'] = nor_pre

In [ ]:
pred_zero['Text'] = data['Text']
pred_zero.to_excel('Fanar-SA-ZeroShot.xlsx', index = False)

In [ ]:
y_true = data['sentiment'].values
print(classification_report(y_true, pred_zero['Normalized Sentiment'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.5498    0.8922    0.6804       167
     Neutral     0.7222    0.0778    0.1405       167
    Positive     0.6745    0.8563    0.7546       167

    accuracy                         0.6088       501
   macro avg     0.6489    0.6088    0.5252       501
weighted avg     0.6489    0.6088    0.5252       501



# Pred Few Shot

In [ ]:
content = '''أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر. صنّف مشاعر الجملة التالية بناءً على نبرتها العاطفية. اختر شعورًا واحدًا فقط من بين: إيجابي، سلبي، أو محايد.

مثال 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

المشاعر المتوقعة: إيجابي

مثال 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت

المشاعر المتوقعة: سلبي

مثال 3:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

المشاعر المتوقعة: محايد
'''

few_pred = []
for i, text in enumerate(data['Text']):
    prompt = f'''الجملة الذي عليك تصنيفها
    الجملة:
    {text}

    المشاعر المتوقعة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    few_pred.append(response)

In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = few_pred
pred_few['Predicted'].value_counts()

,count
Predicted,
سلبي,151
إيجابي,111
محايد\n\nالثقة: 95%,28
محايد,11
يجابي,6
...,...
سلبي\n\nالتفسير: الجملة تحمل نبرة من الألم والشدة، حيث يصف الشخص حالة من المعاناة (مشي على النار)، مما يشير إلى مشاعر سلبية.,1
محايد\n\n(على الرغم من أن الجملة تتحدث عن منافسة بين فريقي كرة القدم، إلا أنها لا تعبر عن مشاعر إيجابية أو سلبية بشكل مباشر. إنها ببساطة تقدم معلومات عن الأداء التنافسي.) ولكن بناءً على السياق العام، يمكن اعتبارها إيجابية لأن المنافسة في الرياضة غالباً ما تعتبر إيجابية. ولكن بناءً على النص فقط، فإن المحايد هو الخيار الأكثر دقة. ومع ذلك، في سياق الرياضة، قد يعتبر البعض هذا الإيجابي. دعنا نلتزم بالمحايد بناءً على النص فقط. ولكن، في سياق الرياضة، يمكن اعتبارها إيجابية.,1
إيجابي! يبدو أن الجملة تعبر عن شعور بالفرح والراحة والاسترخاء بعد يوم طويل، خاصة مع استخدام الرموز التعبيرية التي تعزز هذا الشعور.,1


In [ ]:
nor_pre = []
for pr in pred_few['Predicted']:
  if "سلبي" in pr:
    nor_pre.append("Negative")
  elif "سلبية" in pr:
    nor_pre.append("Negative")
  elif "سلبى" in pr:
    nor_pre.append("Negative")
  elif "Negative" in pr:
    nor_pre.append("Negative")
  elif "إيجابي" in pr:
    nor_pre.append("Positive")
  elif "إيجابية" in pr:
    nor_pre.append("Positive")
  elif "ايجابي" in pr:
    nor_pre.append("Positive")
  elif "ايجابية" in pr:
    nor_pre.append("Positive")
  elif "يجابي" in pr:
    nor_pre.append("Positive")
  elif "محايد" in pr:
    nor_pre.append("Neutral")
  elif "محايدة" in pr:
    nor_pre.append("Neutral")
  elif "محيادي" in pr:
    nor_pre.append("Neutral")
  elif "محاييد" in pr:
    nor_pre.append("Neutral")
  elif "محيود" in pr:
    nor_pre.append("Neutral")
  elif "المحيد" in pr:
    nor_pre.append("Neutral")
  else:
    print(pr)
    nor_pre.append("Unclassified")

pred_few['Normalized Sentiment'] = nor_pre

In [ ]:
pred_few['Text'] = data['Text']
pred_few.to_excel('Fanar-SA-FewShot.xlsx', index = False)

In [ ]:
pred_few['Normalized Sentiment'].value_counts()

,count
Normalized Sentiment,
Negative,254
Positive,194
Neutral,53


In [ ]:
y_true = data['sentiment'].values
print(classification_report(y_true, pred_few['Normalized Sentiment'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.5906    0.8982    0.7126       167
     Neutral     0.6981    0.2216    0.3364       167
    Positive     0.7113    0.8263    0.7645       167

    accuracy                         0.6487       501
   macro avg     0.6667    0.6487    0.6045       501
weighted avg     0.6667    0.6487    0.6045       501



# CoT

In [ ]:
content = '''أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر. صنّف مشاعر الجملة التالية بناءً على نبرتها العاطفية. اختر شعورًا واحدًا فقط من بين: إيجابي، سلبي، أو محايد.

الخطوة 1: قراءة الجملة
اقرأ الجملة بعناية لفهم معناها الكامل وسياقها ونبرتها. خذ بعين الاعتبار التصريحات الصريحة وأي إشارات عاطفية ضمنية.

الخطوة 2: تحديد اللغة العاطفية
- قم بتمييز الكلمات الإيجابية، مثل الكلمات التي تدل على الرضا أو السعادة أو المدح (مثل: رائع، مدهش، أحب، أحسنت).
- قم بتمييز الكلمات السلبية، مثل الكلمات التي تدل على عدم الرضا أو الإحباط أو النقد (مثل: سيء، أكره، مكسور، مخيب).
- العبارات المحايدة لا تتضمن مدحًا أو نقدًا.

الخطوة 3: تحليل التوازن العاطفي
انظر إلى النبرة العامة ونية الجملة، بما في ذلك السخرية أو التناقض إن وجد.

الخطوة 4: تحديد الشعور
- إذا طغت المشاعر الإيجابية، صنفها كـ "إيجابية".
- إذا طغت المشاعر السلبية، صنفها كـ "سلبية".
- إذا لم يكن هناك اتجاه عاطفي واضح أو كانت الجملة معلوماتية بحتة، صنفها كـ "محايدة".

مثال 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

التحليل:
الكلمات الإيجابية: فرحة، أجمل، الخير
الكلمات السلبية: لا يوجد
التوازن العاطفي: التغريدة تستخدم لغة إيجابية فقط وتروج لرسالة مفعمة بالتفاؤل والعطاء.
الشعور: إيجابي

المشاعر المتوقعة: إيجابي

مثال 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت

التحليل:
الكلمات الإيجابية: لا يوجد
الكلمات السلبية: سئمت، دمعة
التوازن العاطفي: التغريدة تحتوي فقط على كلمات سلبية تدور حول التعب والحزن والاجتياح العاطفي.
الشعور: سلبي

المشاعر المتوقعة: سلبي

مثال 3:
إذا لم تستطع أن تترك أثراً جميلاً في القلوب فلا تزرع فيها ألماً لا يُنسى.

التحليل:
الكلمات الإيجابية: جميلاً
الكلمات السلبية: ألماً
التوازن العاطفي: التغريدة تحتوي على إشارات إيجابية وسلبية في آن واحد، ونبرتها العامة وعظية وتحذيرية دون تعبير عاطفي مباشر.
الشعور: محايد

المشاعر المتوقعة: محايد
'''

cot_pred = []
for i, text in enumerate(data['Text']):
    prompt = f'''الجملة الذي عليك تصنيفها
    الجملة:
    {text}'''

    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    cot_pred.append(response)

In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = cot_pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
"بناءً على الجملة، يمكن تصنيف المشاعر بأنها إيجابية. رغم وجود ذكر للنقص في الأهلي، إلا أن النغمة العامة للجملة تعبر عن الثقة والاعتزاز، خاصة مع تكرار كلمة ""كبير"" ثلاث مرات ورمز القلب الأخضر الذي يرمز عادة إلى الحب والإيجابية.",1
"بناءً على اللغة المستخدمة، فإن النبرة العاطفية لهذه الجملة هي سلبية. تشير الكلمات مثل ""ضلم""، ""قمع""، و""قتل"" إلى مشاعر الغضب والإدانة.",1
سلبية\n\nالتحليل:\nالكلمات الإيجابية: لا يوجد\nالكلمات السلبية: وش ذنبه، نهاية الحب، اشغلتونا فيها (بإشارة إلى نهاية الحب كشيء سلبي)، 💔 (رمز القلب المكسور، وهو رمز عاطفي سلبي)\nالتوازن العاطفي: الجملة تعبر عن الاستياء والضيق من نهاية الحب، مع استخدام رمز القلب المكسور الذي يعزز النبرة السلبية.\nالشعور: سلبي,1
"بناءً على اللغة المستخدمة في الجملة، يمكن تصنيف المشاعر بأنها ""سلبية"". حيث تشير الجملة إلى وجود صعوبات ومشاكل مرتبطة ببناء منزل، مما يعكس شعورًا بالإحباط أو الضغط.",1
"بناءً على النبرة العاطفية للجملة، يمكن تصنيفها على أنها ""سلبية"". تشير الرموز التعبيرية 😫 و😭😭 إلى الإحباط والتعاسة، مما يعكس شعورًا سلبيًا عامًا.",1
...,...
"بناءً على النبرة العاطفية، يمكن تصنيف هذه الجملة على أنها ""سلبية"". تحتوي الجملة على علامات الاستفهام والشكوك، بالإضافة إلى استخدام عبارة ""وش تسذبون علينا""، والتي يمكن تفسيرها على أنها تعبير عن الاستياء أو الغضب.",1
"بناءً على الجملة المقدمة، يمكن تصنيف المشاعر بأنها ""محايدة"". حيث أن الجملة تقدم معلومات عن تأثير كوفيد-19 على اقتصادات بعض الدول دون أي تعبير عاطفي واضح.",1
"بناءً على الرموز التعبيرية واللغة المستخدمة، يبدو أن الجملة تحمل شعورًا سلبيًا. الرموز التعبيرية مثل 😭 (دموع) و💔 (قلب مكسور) تشير إلى الحزن أو الألم، بينما الرموز التعبيرية مثل 😱 و🔥 قد تشير إلى الخوف أو الغضب، لكن في هذا السياق، يبدو أنها تستخدم لزيادة التأكيد على الشعور السلبي. الكلمات مثل ""يارب"" و""لو جا اروح"" تعزز الشعور باليأس أو القلق.\n\nالمشاعر المتوقعة: سلبي",1


In [ ]:
nor_pre = []
for pr in pred_cot['Predicted']:
  if "الشعور:" in pr:
      answer = pr.split("الشعور:")[-1].strip()
      if "سلبي" in answer:
        nor_pre.append("Negative")
      elif "سلبية" in answer:
        nor_pre.append("Negative")
      elif "سلبى" in answer:
        nor_pre.append("Negative")
      elif "Negative" in answer:
        nor_pre.append("Negative")
      elif "إيجابي" in answer:
        nor_pre.append("Positive")
      elif "إيجابية" in answer:
        nor_pre.append("Positive")
      elif "ايجابي" in answer:
        nor_pre.append("Positive")
      elif "ايجابية" in answer:
        nor_pre.append("Positive")
      elif "يجابي" in answer:
        nor_pre.append("Positive")
      elif "محايد" in answer:
        nor_pre.append("Neutral")
      elif "محايدة" in answer:
        nor_pre.append("Neutral")
      elif "محيادي" in answer:
        nor_pre.append("Neutral")
      elif "محاييد" in answer:
        nor_pre.append("Neutral")
      elif "محيود" in answer:
        nor_pre.append("Neutral")
      elif "المحيد" in answer:
        nor_pre.append("Neutral")
      else:
        print(answer)
        nor_pre.append("Unclassified")
  else:
        if "سلبي" in pr:
          nor_pre.append("Negative")
        elif "سلبية" in pr:
          nor_pre.append("Negative")
        elif "سلبى" in pr:
          nor_pre.append("Negative")
        elif "Negative" in pr:
          nor_pre.append("Negative")
        elif "إيجابي" in pr:
          nor_pre.append("Positive")
        elif "إيجابية" in pr:
          nor_pre.append("Positive")
        elif "ايجابي" in pr:
          nor_pre.append("Positive")
        elif "ايجابية" in pr:
          nor_pre.append("Positive")
        elif "يجابي" in pr:
          nor_pre.append("Positive")
        elif "محايد" in pr:
          nor_pre.append("Neutral")
        elif "محايدة" in pr:
          nor_pre.append("Neutral")
        elif "محيادي" in pr:
          nor_pre.append("Neutral")
        elif "محاييد" in pr:
          nor_pre.append("Neutral")
        elif "محيود" in pr:
          nor_pre.append("Neutral")
        elif "المحيد" in pr:
          nor_pre.append("Neutral")
        else:
          print(pr)
          nor_pre.append("Unclassified")

pred_cot['Normalized Sentiment'] = nor_pre

In [ ]:
pred_cot['Text'] = data['Text']
pred_cot.to_excel('Fanar-SA-CoT.xlsx', index = False)

In [ ]:
y_true = data['sentiment'].values
print(classification_report(y_true, pred_cot['Normalized Sentiment'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.5118    0.9102    0.6552       167
     Neutral     0.5882    0.0599    0.1087       167
    Positive     0.7112    0.7964    0.7514       167

    accuracy                         0.5888       501
   macro avg     0.6037    0.5888    0.5051       501
weighted avg     0.6037    0.5888    0.5051       501

